### Topic 1: Why Ingestion Exists At All

### Concept, Intuition, Why It Exists

- An LLM only knows two things: what was baked into it during training, and whatever you put in the prompt right now. It has never seen your internal FD policy PDF, today's interest rates, or last week's SOP.
- **RAG** exists to close this gap: instead of retraining a model every time a document changes, you store documents searchably and retrieve relevant pieces at query time, then hand them to the LLM as context.
- **Ingestion is stage zero of RAG** — it takes raw, messy, heterogeneous files (text, CSV, Excel, JSON, PDF) and turns them into a uniform shape that chunking, embedding, and retrieval can all operate on identically.
- Mental model — "the new-product-name problem": the knowledge exists *somewhere* (a PDF on a shared drive), but the model can't use it because it was never **ingested** into anything searchable. Being true and being retrievable are two different things.
- **Garbage in, garbage out applies harder here than anywhere else in the pipeline.** Drop a table, mangle a date, duplicate a stale policy — every later stage (chunking, embedding, retrieval) inherits that damage silently.

### Where Ingestion Sits

Raw files → **Ingestion** (normalize into Documents) → Chunking → Embedding → Vector Store → Retrieval → LLM answer. Every stage downstream of ingestion assumes its input already looks like a `Document`. That assumption only holds because ingestion enforces it.

### How It's Implemented In This Project

- Five source types ingested: structured text reference files, CSV/Excel tables, JSON page exports from a conversion team, and PDFs.
- Each source type gets its own loader module, all returning the same `Document` shape — covered in detail in the next topic.

### Real-World Issues

- **Silent failure**: a broken loader returns an empty string instead of raising — looks like "no relevant info" rather than "ingestion broke."
- **Schema drift**: an upstream file's column/field names change with no notice; code that assumes the old shape breaks silently or returns missing data.
- **Scaling**: eager loading holds every Document in memory at once — fine for 20 files, a crash at 50,000.
- **Cost**: re-embedding unchanged content daily because there's no change-detection.
- **Security/PII**: customer records get embedded into a vector store with no access control inherited from the source system.
- **Encoding bugs**: mixed-language content breaks on a non-UTF-8 default encoding.
- **Monitoring**: nobody notices ingestion volume dropped 90% until a customer complains — track documents-loaded-per-source and alert on anomalies.

### Design Decisions & Trade-offs

- Normalize everything into one `Document` shape immediately, at the loader boundary, rather than letting downstream code branch on source type. Small friction at ingestion time, in exchange for zero format-awareness anywhere else, forever.

### Alternatives

- Hand-rolled loaders (this project) — full control, good for learning, fine for a small number of well-understood sources.
- LangChain/LlamaIndex loaders — community-maintained edge-case handling, more sources supported out of the box.
- Managed ingestion (Unstructured.io, cloud Document Intelligence) — enterprise scale, compliance needs, OCR/table quality beyond a one-off script.

### Common Mistakes

- Treating ingestion as a "run once" script instead of a recurring, idempotent job.
- Not versioning the ingestion contract — an upstream schema change degrades retrieval silently.
- Conflating "no exception thrown" with "data is actually correct."

### Lead-Level Interview Questions

**Q: Why is ingestion a separate stage from chunking, not combined?**
A: Separation of concerns — ingestion's job is "get raw bytes into a normalized shape regardless of source." Chunking's job is "split that text into retrieval-sized pieces." Combined, every loader needs to know about chunk sizes, and every chunking change risks touching every loader.

**Q: Retrieval quality silently degraded over a month. Where do you start debugging?**
A: Work backwards stage by stage, checking *output* not just "did it run." Sample raw ingested Documents first — non-empty, sensible? Then chunks — coherent, not cut mid-sentence? Then embeddings — did the model/version change? Most real degradations trace back to ingestion or chunking, because those stages have the least observability.

**Q: A teammate wants to skip the Document abstraction and pass raw strings everywhere. Response?**
A: Looks simpler for one loader and one consumer. Stops looking simpler the moment you have 4 source types and 3 consumers needing to know *which* source a piece of text came from. Without metadata traveling with text, you lose citation, debugging, and per-source access control.

### Hidden Concepts Worth Knowing

- **Idempotency**: ingestion should be safe to run twice on the same input and produce the same result.
- **ETL vs. ELT**: this pipeline is closer to ELT — load raw-ish Documents first, transform (chunk/embed) afterward — keeping the option to re-chunk later without re-reading every source file.
- **Data contracts**: any schema consumed from an upstream team is an implicit contract. Make it explicit rather than discovering breakage in production.

### Revision Summary

> Ingestion exists because LLMs don't know your private data. It's RAG's first stage: normalize heterogeneous raw files into one uniform shape so every downstream stage can be written once, regardless of source format. Get this stage wrong and every later stage silently inherits the damage.